# P06 · Q-learning and domain randomization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P06_q_learning_and_domain_randomization.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Planning and learning · **Time:** about 60 min

**Skill Hub concepts:** Reinforcement learning; Sim-to-real gap and domain randomization

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P06")

## Map: learning from reward, and surviving the trip to hardware

**Markov decision process:** states $s$, actions $a$, reward $r$, discount $\gamma$. The return is
$G = \sum_k \gamma^k r_k$. **Q-learning** (off-policy temporal difference) learns action values directly:
$$Q(s, a) \leftarrow Q(s, a) + \alpha\left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$
(no bootstrap term when $s'$ is terminal). Act **epsilon-greedy**: random action with probability $\epsilon$, else
$\arg\max_a Q(s, a)$: exploration versus exploitation.

| Family | Learns | Examples | Robot use |
|---|---|---|---|
| Value-based | $Q(s, a)$ | Q-learning, DQN | discrete choices |
| Policy gradient, actor-critic | $\pi_\theta(a \mid s)$ (+ value) | PPO, SAC | continuous control, locomotion in simulation |
| Model-based | dynamics model, then plan | MBPO, Dreamer, MPC on a learned model | sample efficiency |

**Sim-to-real gap:** the simulator's mass, friction, latency, sensor noise and contact model differ from the robot.
**Domain randomization** trains (or tunes) over a distribution of simulator parameters so the result works anywhere
in it, ideally including the real robot. Alternatives: system identification (measure the real parameters) and
fine-tuning on hardware.

In [ ]:
import numpy as np

N, START, GOAL, PITS = 5, (0, 0), (4, 4), {(1, 1), (2, 3), (3, 1)}
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]          # up, down, left, right on grid[row, col]

def s_index(cell):
    return cell[0] * N + cell[1]

def env_step(cell, a):
    """Return (next_cell, reward, done). Walls keep you in place."""
    r, c = cell[0] + MOVES[a][0], cell[1] + MOVES[a][1]
    nxt = (r, c) if 0 <= r < N and 0 <= c < N else cell
    if nxt == GOAL:
        return nxt, 1.0, True
    if nxt in PITS:
        return nxt, -1.0, True
    return nxt, -0.01, False

## Task 1: the Q-learning update and epsilon-greedy action choice

In [ ]:
def q_update(Q, s, a, r, s2, done, alpha=0.5, gamma=0.95):
    """Update Q in place and return the temporal-difference error."""
    raise NotImplementedError  # TODO

def epsilon_greedy(Q, s, eps, rng):
    raise NotImplementedError  # TODO

lab.check("qlearn", q_update, epsilon_greedy)

In [ ]:
def train(episodes=3000, eps=0.2, seed=0):
    rng, Q = np.random.default_rng(seed), np.zeros((N * N, 4))
    for _ in range(episodes):
        cell, done, steps = START, False, 0
        while not done and steps < 100:
            a = epsilon_greedy(Q, s_index(cell), eps, rng)
            nxt, r, done = env_step(cell, a)
            q_update(Q, s_index(cell), a, r, s_index(nxt), done)
            cell, steps = nxt, steps + 1
    return Q

try:
    Q = train()
    cell, path = START, [START]
    while cell != GOAL and len(path) < 30:
        cell, _, _ = env_step(cell, int(np.argmax(Q[s_index(cell)])))
        path.append(cell)
    print("greedy path:", path)
except NotImplementedError:
    print("finish Task 1 first")

## Task 2: a simulator with actuator latency, and domain randomization
Plant: $\ddot x = u / m$, controller $u = -k\,(x + 0.3\,v)$, $\Delta t = 0.02$ s, 200 steps from $x = 1$, $v = 0$.
The actuator applies each command `delay` steps late: keep a first-in-first-out buffer that starts as `delay` zeros;
each step append the new command and apply the one you pop from the front. Update $v \mathrel{+}= u_{applied}/m\,\Delta t$,
then $x \mathrel{+}= v\,\Delta t$, then add $x^2 \Delta t$ to the cost. If $|x| > 1000$ return $10^6$ (diverged).

`pick_gain(gains, masses)` returns the gain with the lowest **mean** cost over the masses.

In [ ]:
def episode_cost(k, m, delay=3, dt=0.02, steps=200):
    raise NotImplementedError  # TODO

def pick_gain(gains, masses):
    raise NotImplementedError  # TODO

lab.check("randomize", episode_cost, pick_gain)

In [ ]:
try:
    gains = np.linspace(1, 150, 150)
    k_nominal = pick_gain(gains, [1.0])
    k_random = pick_gain(gains, np.random.default_rng(0).uniform(0.3, 2.0, 20))
    for m in [0.4, 1.0, 2.0]:
        print(f"mass {m}: nominal-tuned k={k_nominal:.0f} cost {episode_cost(k_nominal, m):.3g} | randomized k={k_random:.0f} cost {episode_cost(k_random, m):.3g}")
except NotImplementedError:
    print("finish Task 2 first")

## Checkpoint: Math
**M1 (Apply).** $Q(s, a) = 0$, $\alpha = 0.5$, $r = 1$, $\gamma = 0.9$, $\max_{a'} Q(s', a') = 2$. New $Q(s, a)$?

**M2 (Apply).** Discounted return of rewards 1, 1, 1 with $\gamma = 0.9$?

**M3 (Remember).** Entries in the Q-table for this 5 x 5 grid with 4 actions?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A grasping policy trained in simulation with one fixed friction value fails on the real gripper.
Best first step?
- a) Train longer in the same simulator
- b) Randomize friction, mass, latency and sensor noise during training
- c) Lower the learning rate

**K2 (Evaluate).** From Task 2: what did domain randomization cost on the nominal plant, what did it buy, and what goes
wrong if you randomize over a range far wider than reality?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p06.h`).
3. In `roboprep/checks/p06.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()